# markoutlib Quickstart

End-to-end walkthrough using synthetic data that mimics a realistic
market with informed and uninformed counterparties.

In [1]:
from datetime import datetime, timedelta

import numpy as np
import polars as pl

import markoutlib as mo

## Synthetic data

We simulate a market with three types of counterparty:

- **HFT_A** and **HFT_B** are informed: they trade in the direction the
  mid is about to move. HFT_A is more aggressive (larger size, higher
  hit rate).
- **Retail** is uninformed: random direction, uncorrelated with future moves.

The mid price follows a mean-reverting process with momentum bursts,
so markout curves will show genuine decay structure.

In [2]:
rng = np.random.default_rng(42)
base = datetime(2024, 1, 15, 10, 0, 0)

# --- quote mid with momentum bursts ---
n_quotes = 50_000  # one quote every 100ms -> ~83 minutes
quote_dt = 0.1  # seconds between quotes

mid = np.empty(n_quotes)
mid[0] = 100.0
drift = 0.0
for i in range(1, n_quotes):
    # occasional momentum shocks
    if rng.random() < 0.003:
        drift = rng.choice([-1, 1]) * rng.uniform(0.004, 0.010)
    # mean-revert the drift quickly so information is absorbed within ~10-15s
    drift *= 0.995
    mid[i] = mid[i - 1] + drift * quote_dt + rng.normal(0, 0.0012)

quote_ts = [base + timedelta(milliseconds=100 * i) for i in range(n_quotes)]
quotes = pl.DataFrame({"timestamp": quote_ts, "mid": mid.tolist()}).cast(
    {"timestamp": pl.Datetime("us")}
)

# --- trades: mix of informed and uninformed ---
n_trades = 2000
trade_indices = np.sort(rng.integers(100, n_quotes - 1000, size=n_trades))

counterparties = rng.choice(
    ["HFT_A", "HFT_B", "Retail"], size=n_trades, p=[0.2, 0.3, 0.5]
)

# short lookahead: information is absorbed within ~10s (100 ticks at 100ms)
lookahead = 100
future_move = np.array(
    [mid[min(j + lookahead, n_quotes - 1)] - mid[j] for j in trade_indices]
)

sides = np.empty(n_trades, dtype=int)
sizes = np.empty(n_trades)
for k in range(n_trades):
    cp = counterparties[k]
    fm = future_move[k]
    if cp == "HFT_A":
        # highly informed: 90% correct direction, large size
        sides[k] = (
            int(np.sign(fm))
            if (rng.random() < 0.90 and fm != 0)
            else rng.choice([-1, 1])
        )
        sizes[k] = rng.lognormal(6.5, 0.6)
    elif cp == "HFT_B":
        # moderately informed: 70% correct direction
        sides[k] = (
            int(np.sign(fm))
            if (rng.random() < 0.70 and fm != 0)
            else rng.choice([-1, 1])
        )
        sizes[k] = rng.lognormal(5.5, 0.8)
    else:
        # retail: coin flip
        sides[k] = rng.choice([-1, 1])
        sizes[k] = rng.lognormal(4.5, 1.0)

half_spread_bps = 1.5
trade_mids = mid[trade_indices]
trade_prices = trade_mids * (1 + sides * half_spread_bps / 10_000)

trades = pl.DataFrame(
    {
        "timestamp": [quote_ts[j] for j in trade_indices],
        "side": sides.tolist(),
        "price": trade_prices.tolist(),
        "mid": trade_mids.tolist(),
        "size": sizes.tolist(),
        "counterparty": counterparties.tolist(),
    }
).cast({"timestamp": pl.Datetime("us")})

print(f"Trades: {trades.shape[0]:,}  |  Quotes: {quotes.shape[0]:,}")
trades.head(5)

Trades: 2,000  |  Quotes: 50,000


timestamp,side,price,mid,size,counterparty
datetime[μs],i64,f64,f64,f64,str
2024-01-15 10:00:10.800,1,100.007597,99.992598,32.493035,"""Retail"""
2024-01-15 10:00:11.500,-1,99.980738,99.995738,450.298397,"""Retail"""
2024-01-15 10:00:17.800,-1,99.991303,100.006304,156.243289,"""HFT_B"""
2024-01-15 10:00:19.200,1,100.031889,100.016887,175.15577,"""Retail"""
2024-01-15 10:00:19.300,-1,99.999481,100.014483,496.631799,"""HFT_B"""


## Compute markouts

Wall-clock horizons from 1 to 60 seconds. We use a single horizon
type here so that all analysis methods work cleanly; trade-clock
and tick-clock are shown at the end.

In [6]:
result = mo.compute(
    trades=trades,
    quotes=quotes,
    horizons=mo.seconds(0, 1, 2, 5, 10, 15, 20, 30, 45, 60),
    unit="bps",
    perspective="maker",
)

## Markout curve

The aggregate markout curve shows how much the mid moves
in the trade's direction at each horizon. A rising curve
that plateaus indicates informed flow.

In [7]:
result.curve()

horizon_type,horizon_value,markout_mean,markout_median,markout_q25,markout_q75,skew,kurtosis,markout_ci_lower,markout_ci_upper,t_stat,p_value,n_obs
str,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64
"""wall""",0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,1.0,2000
"""wall""",1,-0.095508,-0.108903,-0.419868,0.235005,0.038343,0.267221,-0.118259,-0.071585,-8.009662,1.1102e-15,2000
"""wall""",2,-0.199259,-0.22438,-0.751609,0.305738,0.198172,0.435936,-0.24068,-0.159392,-9.57558,0.0,2000
"""wall""",5,-0.514886,-0.598019,-1.708333,0.567983,0.213224,0.116534,-0.602077,-0.421412,-11.624013,0.0,2000
"""wall""",10,-1.044725,-1.200567,-3.112648,0.646284,0.346217,-0.029373,-1.184507,-0.89387,-13.954759,0.0,2000
"""wall""",15,-1.392013,-1.479456,-4.489846,1.111403,0.34826,-0.150365,-1.614637,-1.190913,-13.679276,0.0,2000
"""wall""",20,-1.595978,-1.69746,-5.681768,1.661757,0.317014,-0.290178,-1.837136,-1.33866,-12.438972,0.0,2000
"""wall""",30,-1.829342,-2.167976,-7.254476,2.90319,0.266028,-0.388495,-2.171818,-1.456513,-10.234315,0.0,2000
"""wall""",45,-1.86823,-2.509898,-9.193683,4.782679,0.207899,-0.353344,-2.384492,-1.357175,-7.876924,3.3307e-15,2000


In [8]:
result.plot.curve()

## Segmentation by counterparty

Breaking the curve by counterparty reveals who is informed.
HFT_A should show the steepest rise (most toxic), Retail
should be flat around zero.

In [9]:
result.curve(by="counterparty")

horizon_type,horizon_value,counterparty,markout_mean,markout_median,markout_q25,markout_q75,skew,kurtosis,markout_ci_lower,markout_ci_upper,t_stat,p_value,n_obs
str,i64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64
"""wall""",0,"""Retail""",0.0,0.0,0.0,-0.0,NaN,NaN,0.0,0.0,0.0,1.0,999
"""wall""",0,"""HFT_B""",0.0,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,1.0,633
"""wall""",0,"""HFT_A""",0.0,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,1.0,368
"""wall""",1,"""Retail""",0.003256,-0.009092,-0.322879,0.355425,0.015822,0.183779,-0.026809,0.03295,0.197672,0.843302,999
"""wall""",1,"""HFT_B""",-0.159902,-0.143261,-0.476868,0.17051,-0.007197,0.471255,-0.195715,-0.118919,-7.804457,5.9952e-15,633
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""wall""",45,"""HFT_B""",-3.696533,-3.938757,-10.70197,2.039413,0.409646,-0.002657,-4.492289,-2.885171,-8.466741,0.0,633
"""wall""",45,"""HFT_A""",-4.625548,-4.534491,-11.093749,2.19276,0.071697,-0.394312,-5.768057,-3.423044,-7.385328,1.5210e-13,368
"""wall""",60,"""Retail""",0.378047,0.768367,-7.994823,8.48195,-0.024973,-0.428207,-0.373143,1.188253,0.95515,0.339502,999


In [10]:
result.plot.curve(by="counterparty")

## Heatmap

A heatmap gives a compact view of markout levels
across counterparties and horizons.

In [11]:
result.plot.heatmap(by="counterparty")

## Distribution at a single horizon

Look at the full distribution of per-trade markouts
at the 10-second horizon, split by counterparty.

In [12]:
result.plot.distribution(horizon=mo.seconds(10), by="counterparty")

## Comparison panels

Side-by-side markout curves, one panel per counterparty.

In [13]:
result.plot.comparison(by="counterparty")

## Scatter: size vs markout

Check whether larger trades carry more information.

In [14]:
result.plot.scatter(x="size", horizon=mo.seconds(10))

## Weighted vs unweighted

Compare simple-mean markouts with size-weighted markouts.
If informed traders trade larger, weighting amplifies
the signal.

In [15]:
result.compare(weight="size")

horizon_type,horizon_value,markout_unweighted,markout_weighted,n_obs
str,i64,f64,f64,i64
"""wall""",0,0.0,0.0,2000
"""wall""",1,-0.095508,-0.173044,2000
"""wall""",2,-0.199259,-0.356773,2000
"""wall""",5,-0.514886,-0.878881,2000
"""wall""",10,-1.044725,-1.684113,2000
"""wall""",15,-1.392013,-2.222448,2000
"""wall""",20,-1.595978,-2.549498,2000
"""wall""",30,-1.829342,-2.974968,2000
"""wall""",45,-1.86823,-3.172431,2000


## Statistical tests

Permutation test: is each counterparty's markout
significantly different from the rest?

In [16]:
result.test("counterparty")

segment,segment_n_obs,segment_mean,complement_mean,test_stat,test_p_value
str,i64,f64,f64,f64,f64
"""Retail""",9990,0.09162,-2.12408,2.2157,0.000999
"""HFT_B""",6330,-1.931583,-0.59399,-1.337593,0.000999
"""HFT_A""",3680,-2.455196,-0.693115,-1.762081,0.000999


In [17]:
result.test("counterparty", pairwise=True)

segment_a,segment_b,diff_mean,test_stat,test_p_value_raw,test_p_value_bh
str,str,f64,f64,f64,f64
"""Retail""","""HFT_B""",2.023202,2.023202,0.000999,0.000999
"""Retail""","""HFT_A""",2.546816,2.546816,0.000999,0.000999
"""HFT_B""","""HFT_A""",0.523613,0.523613,0.000999,0.000999


## Spread decomposition

The identity: **effective spread = realized spread + price impact**.
Price impact is the portion of the spread captured by informed traders.

In [18]:
pl.concat([result.spread_decomposition(horizon=mo.seconds(h)) for h in (10, 30, 60)])

horizon_type,horizon_value,effective_spread_mean,effective_spread_median,realized_spread_mean,realized_spread_median,price_impact_mean,price_impact_median,n_obs
str,f64,f64,f64,f64,f64,f64,f64,u32
"""wall""",10.0,1.5,1.5,0.455275,0.299433,1.044725,1.200567,2000
"""wall""",30.0,1.5,1.5,-0.329342,-0.667976,1.829342,2.167976,2000
"""wall""",60.0,1.5,1.5,-0.133439,-0.402012,1.633439,1.902012,2000


In [19]:
pl.concat(
    [
        result.spread_decomposition(horizon=mo.seconds(h), by="counterparty")
        for h in (10, 30, 60)
    ]
)

horizon_type,horizon_value,counterparty,effective_spread_mean,effective_spread_median,realized_spread_mean,realized_spread_median,price_impact_mean,price_impact_median,n_obs
str,f64,str,f64,f64,f64,f64,f64,f64,u32
"""wall""",10.0,"""Retail""",1.5,1.5,1.497804,1.393445,0.002196,0.106555,999
"""wall""",10.0,"""HFT_B""",1.5,1.5,-0.39249,-0.382236,1.89249,1.882236,633
"""wall""",10.0,"""HFT_A""",1.5,1.5,-0.916602,-0.530856,2.416602,2.030856,368
"""wall""",30.0,"""Retail""",1.5,1.5,1.660852,1.530795,-0.160852,-0.030795,999
"""wall""",30.0,"""HFT_B""",1.5,1.5,-2.028194,-2.527471,3.528194,4.027471,633
"""wall""",30.0,"""HFT_A""",1.5,1.5,-2.809859,-2.780686,4.309859,4.280686,368
"""wall""",60.0,"""Retail""",1.5,1.5,1.878047,2.268367,-0.378047,-0.768367,999
"""wall""",60.0,"""HFT_B""",1.5,1.5,-1.74179,-2.330871,3.24179,3.830871,633
"""wall""",60.0,"""HFT_A""",1.5,1.5,-2.827427,-2.953568,4.327427,4.453568,368


## Half-life estimation

Fit an exponential decay to estimate how quickly
information is incorporated into the price.

In [20]:
hl = result.half_life()
if hl.converged:
    print(f"Half-life:  {hl.half_life:.1f}s")
    print(f"Terminal:   {hl.terminal_markout:.2f} bps")
    print(f"R-squared:  {hl.r_squared:.3f}")
else:
    print("Decay fit did not converge")

Half-life:  8.0s
Terminal:   -1.84 bps
R-squared:  0.975


## Raw data export

The underlying per-trade markout data is always
accessible as a Polars (or Pandas) DataFrame.

In [21]:
result.to_polars().head(10)

timestamp,side,price,mid,size,counterparty,future_mid,markout,horizon_type,horizon_value
datetime[μs],i64,f64,f64,f64,str,f64,f64,str,i32
2024-01-15 10:00:10.800,1,100.007597,99.992598,32.493035,"""Retail""",99.992598,-0.0,"""wall""",0
2024-01-15 10:00:11.500,-1,99.980738,99.995738,450.298397,"""Retail""",99.995738,0.0,"""wall""",0
2024-01-15 10:00:17.800,-1,99.991303,100.006304,156.243289,"""HFT_B""",100.006304,0.0,"""wall""",0
2024-01-15 10:00:19.200,1,100.031889,100.016887,175.15577,"""Retail""",100.016887,-0.0,"""wall""",0
2024-01-15 10:00:19.300,-1,99.999481,100.014483,496.631799,"""HFT_B""",100.014483,0.0,"""wall""",0
2024-01-15 10:00:20.100,-1,99.997598,100.0126,62.023321,"""Retail""",100.0126,0.0,"""wall""",0
2024-01-15 10:00:25.200,-1,99.997855,100.012856,995.107548,"""HFT_B""",100.012856,0.0,"""wall""",0
2024-01-15 10:00:25.600,1,100.022557,100.007556,8.922856,"""Retail""",100.007556,-0.0,"""wall""",0
2024-01-15 10:00:27.400,-1,99.994406,100.009407,93.516209,"""Retail""",100.009407,0.0,"""wall""",0


## Alternative clocks: trade-clock and tick-clock

Wall-clock measures "N seconds after the trade". But you can also
measure in **trade-clock** (N trades later) or **tick-clock** (N
quote updates later). Each tells a different story about how
quickly the market processes information.

In [22]:
result_trade = mo.compute(
    trades=trades,
    quotes=quotes,
    horizons=mo.trades(1, 2, 5, 10, 25, 50, 100),
    unit="bps",
    perspective="maker",
)
result_trade.plot.curve()

In [23]:
result_tick = mo.compute(
    trades=trades,
    quotes=quotes,
    horizons=mo.ticks(1, 5, 10, 25, 50, 100, 200),
    unit="bps",
    perspective="maker",
)
result_tick.plot.curve()